# 01 - Build Dataset

Builds the merged, cleaned, region- and year-annotated analysis dataset
for "Trends in Responsa from the 7th Century to the Present."

**Dating rule (`interpolated_year`):** the first responsum in any volume
by an author is assumed written birth+30; the last is assumed written in
the author's death year. Volumes are interpolated independently (not
chained/sequenced), since volumes may be arranged by topic rather than
chronology. A 60-year lifespan is assumed if death_year is missing.

**True position within a volume:** "first" and "last" are recovered from
the `unit` column, which records each responsum's siman (paragraph)
number (or occasionally chapter/part/sermon/etc.) as printed in the
original volume -- NOT from raw row order. It turns out the raw data's
row order is overwhelmingly sorted by word_length instead of true volume
position (correlation ~0.97-1.0 across large volumes), which would
otherwise systematically date an author's shortest responsa to early in
life and longest to near death, dataset-wide. See `parse_unit()` below.

**No-signal responsa:** about 3.2% of rows have `unit` text with no
decodable position at all (a bare section label, an indexing scheme
`parse_unit()` doesn't recognize, or no `unit` text). These are dispersed
with an independent Uniform(0,1) draw (fixed seed, reproducible) across
the author's active range, rather than all placed at one shared position.
Collapsing them onto a single point (e.g. the midpoint) can badly distort
the record for any author who has many such responsa -- e.g. an author
whose volumes are indexed by plain letter/item number rather than siman
number pools hundreds of responsa onto one artificial year, visibly
distorting Figure 1's smoothed curves around that year.

**Exception:** one entry, "תשובות הגאונים" (birth_year=850), is a modern
anthology of several distinct 19th-century scholarly editions of Geonic
responsa, not an individual -- see `DATE_RANGE_OVERRIDE` in the code cell
below for its date range (750-1038 CE, skewed toward the later end) and
the sourced reasoning behind it.

Run this in Google Colab. It expects your two raw metadata files at:

    /content/gdrive/MyDrive/Responsa/Lengths/data/raw/full_stats_with_missing.xlsx
    /content/gdrive/MyDrive/Responsa/Lengths/data/raw/geonim_len_stats.xlsx

and writes the merged dataset to:

    /content/gdrive/MyDrive/Responsa/Lengths/data/processed/bi_plus_geonim.csv

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

BASE = "/content/gdrive/MyDrive/Responsa/Lengths"
RAW = f"{BASE}/data/raw"
OUT = f"{BASE}/data/processed"

import os
os.makedirs(OUT, exist_ok=True)

In [ ]:
import re

import numpy as np
import pandas as pd

MIN_AGE = 30  # first responsum in a volume assumed written this many years after birth

# ---------------------------------------------------------------------------
# Decode the `unit` column into a true position within its volume.
#
# `unit` records how each responsum was labeled in the original printed
# volume -- almost always a siman (paragraph) number in Hebrew numerals
# (e.g. "סימן שכח" = Siman 328), occasionally a chapter/part/sermon/etc.
# number instead, and sometimes no number at all (front matter like a
# preface, back matter like addenda, or a bare section label). This
# replaces raw row order, which turns out to track word_length (the data's
# actual sort order) rather than true volume position -- see the markdown
# cell above.
# ---------------------------------------------------------------------------
GEMATRIA = {
    "א": 1, "ב": 2, "ג": 3, "ד": 4, "ה": 5, "ו": 6, "ז": 7, "ח": 8, "ט": 9,
    "י": 10, "כ": 20, "ך": 20, "ל": 30, "מ": 40, "ם": 40, "נ": 50, "ן": 50,
    "ס": 60, "ע": 70, "פ": 80, "ף": 80, "צ": 90, "ץ": 90,
    "ק": 100, "ר": 200, "ש": 300, "ת": 400,
}


def _gematria(letters):
    return sum(GEMATRIA.get(ch, 0) for ch in letters)


# Word prefixes that introduce a genuine, decodable position number.
_NUMBERED_PREFIXES = [
    "סימן", "פרק", "חלק", "כרך", "דרוש", "מאמר", "שאלה", "קונטרס",
]
# `unit` text indicating front matter (belongs at the very start of a volume).
_FRONT_MATTER_MARKERS = ["הקדמה", "מבוא", "פתיחה", "דברי פתיחה", "תוכן העניינים", "תוכן עניינים"]
# `unit` text indicating back matter (belongs at the very end of a volume).
_BACK_MATTER_MARKERS = ["מילואים", "נספח", "השמטות", "קונטרס אחרון", "הערות", "הגהות", "סיום"]


def parse_unit(unit):
    """Decode a `unit` string into (order_key, category).

    order_key is a float used to rank responsa within a volume:
    -inf for front matter, +inf for back matter, the decoded number for
    anything with a genuine siman/chapter/etc. number, and NaN when no
    position can be recovered at all (handled separately downstream).
    """
    if not isinstance(unit, str):
        return (np.nan, "null")
    s = unit.strip()
    for prefix in _NUMBERED_PREFIXES:
        m = re.match(rf"^{prefix}\s+([א-ת]+)", s)
        if m:
            return (float(_gematria(m.group(1))), "numbered")
    m = re.match(r"^מערכת\s+([א-ת]+)\s+אות\s+([א-ת]+)", s)
    if m:
        # alphabetical-index scheme: combine the two letters into one key
        return (float(_gematria(m.group(1)) * 1000 + _gematria(m.group(2))), "numbered")
    if any(k in s for k in _FRONT_MATTER_MARKERS):
        return (-np.inf, "front_matter")
    if any(k in s for k in _BACK_MATTER_MARKERS):
        return (np.inf, "back_matter")
    return (np.nan, "unrecognized")


# ---------------------------------------------------------------------------
# 1. Load and concatenate the two raw metadata files
# ---------------------------------------------------------------------------
df = pd.read_excel(f"{RAW}/full_stats_with_missing.xlsx")
df_geonim = pd.read_excel(f"{RAW}/geonim_len_stats.xlsx")

df["source"] = "dataset_1"       # Bar-Ilan Responsa Project (Rishonim/Acharonim)
df_geonim["source"] = "dataset_2"  # Geonim-era responsa

df = pd.concat([df, df_geonim], ignore_index=True)

# True position of each responsum within (author, book), decoded from the
# printed siman/chapter/etc. number in `unit` -- see parse_unit() above.
# Replaces the old row-order-based unit_index, which turned out to track
# word_length (the data's actual sort order) rather than volume position.
_parsed = df["unit"].apply(parse_unit)
df["order_key"] = _parsed.apply(lambda t: t[0])
df["order_category"] = _parsed.apply(lambda t: t[1])

# Computed once, vectorized, over the whole dataframe (rather than inside
# the per-author groupby below) to avoid a pandas groupby.apply quirk where
# a group containing only a single book collapses the returned Series into
# a malformed one-row DataFrame. Grouped by (author_name, birth_year,
# book_name) -- birth_year is included because one author_name in this
# dataset ("תשובות הגאונים") covers two distinct birth-year entries that
# happen to share book_name text; keeping birth_year in the key prevents
# their responsa from being ranked against each other.
_group_cols = ["author_name", "birth_year", "book_name"]
_orderable_mask = df["order_category"].isin(["numbered", "front_matter", "back_matter"])

# Responsa with no decodable position at all ("unrecognized" unit text, or
# no unit text/"null") get an independent Uniform(0,1) relative position --
# spreading them across the author's full active period -- rather than a
# single shared value. A fixed seed keeps this reproducible. Drawing
# per-row from a distribution (rather than ranking these rows by anything
# derived from the raw data, e.g. row order) is deliberate: row order is
# known to track word_length (see the markdown cell above), so ranking by it
# would reintroduce exactly the length-vs-position bias parse_unit() was
# built to avoid.
_DISPERSE_SEED = 20260907
_disperse_rng = np.random.default_rng(_DISPERSE_SEED)
_non_orderable_mask = ~_orderable_mask
relative_position_raw = pd.Series(0.5, index=df.index)
relative_position_raw.loc[_non_orderable_mask] = _disperse_rng.uniform(
    0.0, 1.0, size=int(_non_orderable_mask.sum())
)
_orderable = df[_orderable_mask]
_ranks = _orderable.groupby(_group_cols, dropna=False)["order_key"].rank(method="first")
_sizes = _orderable.groupby(_group_cols, dropna=False)["order_key"].transform("size")
relative_position_raw.loc[_orderable.index] = np.where(
    _sizes > 1, (_ranks - 1) / (_sizes - 1).clip(lower=1), 0.5
)
df["relative_position_raw"] = relative_position_raw

# ---------------------------------------------------------------------------
# 2. Manually fill in birth year / country for ~45 authors missing metadata
#    (hand-collected by the authors; see Summary Statistics.ipynb cell 5)
# ---------------------------------------------------------------------------
manual_fill = {
    "ר' יעקב ישראל חגיז": (1620, "ארץ ישראל"),
    "ר' יוסף חיים אל חכם": (1851, "עיראק"),
    "ר' אריה ליבוש ליפשיץ": (1760, "פולין"),
    "ר' משה יהושע יהודה לייב דיסקין": (1817, "ליטא, ארץ ישראל"),
    "ר' יוסף משאש": (1892, "מרוקו, ישראל"),
    "ר' יהודה אריה גרוסנס": (1870, "פולין"),
    "ר' פסח אליהו פלק": (1878, "פולין"),
    "ר' משה גרינוולד": (1853, "הונגריה"),
    "ר' אריה צבי פרומר": (1884, "פולין"),
    "ר' יואב יהושע וינגרטן": (1845, "פולין"),
    "ר' שמואל יצחק שור": (1873, "פולין"),
    "ר' ברוך לאווסקי": (1867, "פולין"),
    "ר' מאיר די בוטון": (1575, "טורקיה"),
    "ר' מרדכי זיסקינד רוטנברג": (1830, "פולין"),
    "ר' שמואל ענגיל": (1853, "פולין"),
    "ר' יוסף אליהו הנקין": (1881, 'בלארוס, ארה"ב'),
    "ר' שלום משאש": (1909, "מרוקו, ישראל"),
    "ר' יהודה הרצל הנקין": (1945, 'ארה"ב, ישראל'),
    "ר' יצחק אבולעפיה": (1824, "ארץ ישראל"),
    "ר' שמואל אבוהב": (1610, "איטליה"),
    "ר' שלמה יהודה טאבאק": (1832, "הונגריה"),
    "ר' שלמה זלמן ליפשיץ": (1765, "פולין"),
    "ר' יעקב אבן יחיא": (1470, "פורטוגל"),
    "ר' יחיאל באסן": (1653, "איטליה"),
    "ר' סיני ספיר": (1820, "ליטא"),
    "ר' יחזקאל קצנלנפוגן": (1668, "פולין"),
    "ר' יוסף אירגאס": (1685, "איטליה"),
    "ר' אברהם פיוטרקובסקי": (1897, "פולין"),
    "ר' רפאל כץ": (1770, "פולין"),
    "ר' יעקב מאיר פאדווא": (1890, "הונגריה, אנגליה"),
    "ר' אברהם ליטש רוזנבוים": (1860, "הונגריה"),
    "ר' ישראל בעלסקי": (1938, 'ארה"ב'),
    "ר' יעקב ברוכין": (1879, "פולין"),
    "ר' מרדכי פארהאנד": (1872, "הונגריה"),
    "ר' יחיאל העליר": (1814, "הונגריה"),
    "ר' אברהם ישראל זאבי": (1869, "פולין"),
    "ר' אליהו בקשי דורון": (1941, "ישראל"),
    "ר' ישראל ניסן קופרשטוך": (1900, "פולין"),
    "ר' בן ציון בלום": (1908, "הונגריה"),
    "ר' אהרן לפפא": (1790, "פולין"),
    'ר\' חיים ב"ר שמחה הכהן רפפורט': (1700, "פולין"),
    "ר' משה שטרן": (1914, 'הונגריה, ארה"ב'),
    'ר\' ראובן ב"ר בצלאל הכהן רפפורט': (1750, "פולין"),
    "ר' אריה יהודה ליב תאומים": (1813, "הונגריה"),
    'ר\' שלמה ב"ר יצחק לבית לוי': (1020, "צרפת"),
    "ר' מאיר אריק": (1855, "פולין"),
}

country_translation = {
    "ארץ ישראל": "Israel",
    "עיראק": "Iraq",
    "פולין": "Poland",
    "ליטא": "Lithuania",
    "ליטא, ארץ ישראל": "Lithuania",
    "מרוקו, ישראל": "Morocco",
    "מרוקו": "Morocco",
    "הונגריה": "Hungary",
    "איטליה": "Italy",
    'בלארוס, ארה"ב': "United States",
    'ארה"ב': "United States",
    'ארה"ב, ישראל': "Israel",
    "פורטוגל": "Portugal",
    "טורקיה": "Turkey",
    "הונגריה, אנגליה": "Hungary",
    'הונגריה, ארה"ב': "Hungary",
    "צרפת": "France",
}

for name in manual_fill:
    birth, loc = manual_fill[name]
    manual_fill[name] = (birth, country_translation.get(loc, loc))


def fill_missing(row):
    if pd.isna(row["birth_year"]) and row["author_name"] in manual_fill:
        row["birth_year"] = manual_fill[row["author_name"]][0]
    if pd.isna(row["country"]) and row["author_name"] in manual_fill:
        row["country"] = manual_fill[row["author_name"]][1]
    return row


df = df.apply(fill_missing, axis=1)

# ---------------------------------------------------------------------------
# 3. Interpolate a year of authorship for each responsum.
#    First responsum in a volume -> birth+MIN_AGE. Last responsum in a
#    volume -> death. Each volume is interpolated independently (volumes
#    are NOT chained/sequenced relative to one another). A 60-year
#    lifespan is assumed if death_year is missing. See the markdown cell above.
#
#    Grouped by (author_name, birth_year) rather than author_name alone:
#    one author_name in the raw data -- "תשובות הגאונים" -- actually
#    covers two distinct entries with different recorded birth years
#    (800 and 850); grouping by author_name alone would silently apply
#    only the first-encountered birth year to every row sharing that
#    name. This is the only author_name in the dataset with more than
#    one distinct birth_year, so this change affects nothing else.
#
#    DATE_RANGE_OVERRIDE below replaces the birth+MIN_AGE/death (or
#    birth+60 default) window for specific (author_name, birth_year)
#    entries that are not individual people:
#
#    "תשובות הגאונים" (birth_year=850, no death_year recorded): this
#    entry is not one person -- it's five 19th-century scholarly
#    editions collecting responsa from many different historical Geonim
#    (Sha'arei Tzedek; Cassel's Geonim Kadmonim; Sha'arei Teshuva; the
#    Musafia and Coronel editions; Harkavy's corpus; and Geonei Mizrach
#    U'Maarav). The named Geonim represented across these editions span
#    roughly Yehudai Gaon (mid-8th century) to Hai Gaon (d. 1038 CE, the
#    last major Gaon). More than half of all surviving Geonic responsa
#    come from the final generation -- Sherira Gaon and his son Hai
#    Gaon, who led the Pumbedita academy from roughly 968-1038 CE -- so
#    the true distribution is weighted heavily toward that later end,
#    not spread evenly across the full 750-1038 window. (The separate
#    "תשובות רב נטרונאי גאון" entry, birth_year=800, is a real
#    individual -- Natronai ben Hilai, Gaon of Sura c. 857-865 -- and
#    keeps the standard birth+30/birth+60 treatment.)
# ---------------------------------------------------------------------------
DATE_RANGE_OVERRIDE = {
    # (author_name, birth_year): (start_year, end_year, skew)
    # skew is the exponent applied to each responsum's relative position
    # within its volume before mapping onto [start_year, end_year]. skew=1
    # is the same plain linear interpolation used everywhere else; skew<1
    # pushes more responsa toward end_year. skew=0.4 is chosen so the
    # MEDIAN interpolated year lands at 968 (the start of Sherira Gaon's
    # tenure) -- i.e. roughly half the responsa fall in the last ~70 of
    # these 288 years, matching the "more than half from Sherira/Hai" fact
    # above (median of Uniform(0,1)**0.4 is 0.5**0.4 ~= 0.76, and
    # 750 + 0.76*(1038-750) ~= 968).
    ("תשובות הגאונים", 850.0): (750, 1038, 0.4),
}


def assign_interpolated_year(group):
    author = group["author_name"].iloc[0]
    birth = group["birth_year"].iloc[0]
    death = group["death_year"].iloc[0]

    if pd.isna(birth):
        group["interpolated_year"] = np.nan
        return group

    override = DATE_RANGE_OVERRIDE.get((author, birth))
    if override is not None:
        start_year, end_year, skew = override
    else:
        if pd.isna(death):
            death = birth + 60
        start_year = birth + MIN_AGE
        end_year = death
        skew = 1.0

    # True position within (author, book), decoded from `unit` rather than
    # raw row order -- computed once for the whole dataframe above (see
    # relative_position_raw), since a per-group groupby.apply here hits a
    # pandas edge case for single-book groups.
    relative_position = group["relative_position_raw"].astype(float) ** skew

    group["interpolated_year"] = start_year + relative_position * (end_year - start_year)
    return group


df = df.groupby(["author_name", "birth_year"], dropna=False, group_keys=False).apply(assign_interpolated_year)
df["year"] = df["interpolated_year"]
df["age_at_death"] = df["death_year"] - df["birth_year"]

# ---------------------------------------------------------------------------
# 4. Clean word_length; drop citation/footnote-only entries
#    (observations whose book name marks them as a note/gloss, e.g. הערות)
# ---------------------------------------------------------------------------
df["word_length"] = pd.to_numeric(df["word_length"], errors="coerce")
df_filtered = df[~df["book_name"].str.contains("הערות|הגהות", na=False)].copy()

# ---------------------------------------------------------------------------
# 5. Assign a geographic region from country
# ---------------------------------------------------------------------------
region_map = {
    "Eastern Europe": [
        "Russia", "Poland", "Ukraine", "Lithuania", "Latvia", "Estonia",
        "Belarus", "Hungary", "Czech Republic", "Slovakia", "Romania", "Bulgaria",
    ],
    "Central and Western Europe": [
        "France", "Germany", "Netherlands", "Belgium", "Switzerland",
        "Austria", "United Kingdom", "England", "Ireland",
    ],
    "Italy and the Balkans": [
        "Italy", "Greece", "Yugoslavia", "Bosnia", "Serbia", "Croatia", "Bosnia-Herzegovina",
    ],
    "Iberia": ["Spain", "Portugal"],
    "North Africa": ["Morocco", "Algeria", "Tunisia", "Libya", "Egypt"],
    "Israel/Palestine": [
        "Israel", "Palestine", "Ottoman Palestine", "Mandatory Palestine",
        "Eretz Israel", "ישראל",
    ],
    "West Asia (ex Israel)": [
        "Saudi Arabia", "Yemen", "Oman", "United Arab Emirates", "Arabia", "Hejaz",
        "Syria", "Turkey", "Iraq", "Iran", "Mesopotamia", "Persia",
        "Lebanon", "Jordan", "Qatar", "Bahrain", "Kuwait", "Georgia", "Armenia", "Azerbaijan",
    ],
    "North America/Australia": ["United States", "USA", "Canada", "Mexico", "Australia"],
}


def map_region(row):
    country = row["country"]
    for region, countries in region_map.items():
        if country in countries:
            return region
    if pd.isna(country) and row["source"] == "dataset_2":
        # Geonim-era authors have no country recorded in geonim_len_stats.xlsx;
        # the Geonim were centered in Babylonia (modern Iraq).
        return "West Asia (ex Israel)"
    return "Other"


df_filtered["region"] = df_filtered.apply(map_region, axis=1)

# ---------------------------------------------------------------------------
# 6. Save
# ---------------------------------------------------------------------------
import os
os.makedirs(OUT, exist_ok=True)
df_filtered.to_csv(f"{OUT}/bi_plus_geonim.csv", index=False)
print(f"Wrote {OUT}/bi_plus_geonim.csv  ({len(df_filtered):,} rows)")
print("Note: this file still includes observations with word_length <= 10")
print("(short citations/footnotes not yet excluded -- see 02_summary_statistics.py")
print("and the paper's Section III.A, 'Excluding Non-Responsa').")